# Step 5 – Ice-velocity overlay on the Pauli and Cloude–Pottier images

Adds the **annual ice velocity** (`Antarctica_ice_velocity_2024_2025_1km_v01.2.nc`) on top of the images made in Step 3 (Pauli) and Step 4 (Cloude–Pottier), **on ice shelves**:

| Layer | What you see |
|---|---|
| Base image | Pauli RGB, or the Cloude–Pottier maps (H, α, … you choose) |
| Light colour layer | ice speed (m/yr), semi-transparent, on **ice shelves** |
| Arrows | flow direction (all the same length, so they show direction, not speed) |

**How the projection is handled.** The result `.tif` files of Steps 3 and 4 carry ground control points (GCPs). Here they are used to warp each image to the polar stereographic grid of the velocity data (**EPSG:3031**), so the two line up exactly. The ice-shelf outline comes from the MEaSUREs Antarctic Boundaries (NSIDC-0709, `IceShelf_Antarctica_v02.shp`); velocity on grounded ice and ocean is not drawn. The outline is from 2008–2009 and is applied generously (any 1 km cell touching a shelf polygon is kept), so a thin strip of grounded ice or of an old front position may show. Velocity is also only drawn where the image has data.

**Good to know.** The velocity map is a 2024–2025 mean, the BIOMASS scenes are acquired in 2026. The overlay shows the *typical* flow, not the flow on the day of the acquisition.

**Output**: `<OUTPUT_DIR>/<date_time>_velocity_overlay/<stack folder>/L1C/<product>/<product>_pauli_velocity.png` and `<product>_h_alpha_velocity.png`, plus a `run_info.json`. Earlier results are never overwritten.

**How to use**
0. First time only: run **5.0**, then restart the kernel.
1. Check the settings in **5.1** (the defaults cover all seven stacks).
2. Run **5.2** (tools) and **5.3** (checks: shows what was found).
3. Run **5.4** (Pauli) and/or **5.5** (Cloude–Pottier).

Steps 3 and 4 must have been run first, with `SAVE_GEOTIFF = True`.

## 5.0 First time only: install the packages
Run once, then restart the kernel. Skip this cell if 5.2 runs without errors.

In [ ]:
%pip install --user rasterio xarray netCDF4 scipy matplotlib geopandas shapely pyogrio

## 5.1 Settings
**Which data**
- `STACKS`: the stack / product folders to process (the names of the folders in `biomass_inputs`). `None` = every result found.
- `OUTPUT_DIR`: the same main folder you used in Steps 3 and 4 (this is where the `.tif` results are read, and where the overlays are written).
- `RUN_FOLDERS`: `None` = use the **newest** Step 3/4 result of every scene, from any run. Or give part of a run-folder name, e.g. `["window200"]`.

**Input files** (leave as `None` to search automatically in `~/data` (including sub-folders), `~`, and the folder of this notebook)
- `VELOCITY_FILE`: the velocity NetCDF.
- `ICE_SHELF_FILE`: the MEaSUREs ice-shelf shapefile `IceShelf_Antarctica_v02.shp` (from NSIDC-0709; keep the `.dbf`, `.shx`, `.prj` next to it). Used to keep ice shelves only.

**What to make**
- `DO_PAULI`, `DO_CLOUDE_POTTIER`: switch each analysis on or off.
- `CP_PANELS`: which Cloude–Pottier maps to show, from `"span"`, `"H"`, `"A"`, `"alpha"`, `"beta"`, `"beta1"`.

**Look of the overlay**
- `VEL_ALPHA`: transparency of the colour layer (0 = invisible, 1 = solid).
- `VEL_MAX`: speed (m/yr) at the top of the colour scale. `None` = automatic per figure; set a number to make all figures comparable.
- `ARROW_SPACING_KM`: distance between arrows. `ARROW_LENGTH_KM`: length of each arrow.
- `ARROWS_SCALED_BY_SPEED`: `False` = equal-length arrows (direction only); `True` = length grows with speed.
- `MAX_PIXELS`: the images are reduced to at most this many pixels on the long side (keeps the kernel from running out of memory).

In [ ]:
# ---------- Which data ----------
STACKS = [
    "stack_02",
    "stack_27",
    "stack_40",
    "stack_43",
    "stack_20",
    "BIO_S1_STA__1S_20261004T060510_20261004T060538_T_G01_M05_C07_T033_F250_02_DZ31YD",
    "BIO_S2_STA__ID_20260514T160710_20260601T160741_T_G01_M03_S0709_E0126_D_02_DT479Y",
    "BIO_S1_STA__ID_20260704T161134_20260722T161209_T_G01_M04_S0710_E0111_D_02_DV87RS",
]                                   # None = everything found
OUTPUT_DIR = "~/amorgan/biomass_output"     # same folder as in Steps 3 and 4
RUN_FOLDERS = None                  # None = newest result of every scene

# ---------- Input files (None = search automatically) ----------
VELOCITY_FILE = "~/amorgan/Antarctica_ice_velocity_2024_2025_1km_v01.2.nc"
ICE_SHELF_FILE = "~/amorgan/data/IceShelf_Antarctica_v02.shp"   # MEaSUREs ice shelves (NSIDC-0709)

# ---------- What to make ----------
DO_PAULI = True
DO_CLOUDE_POTTIER = True
CP_PANELS = ["H", "alpha"]          # span, H, A, alpha, beta, beta1

# ---------- Look of the overlay ----------
ARROW_SPACING_KM = 4                # distance between arrows (km)
ARROW_MAX_SPEED = None              # m/yr that gets the longest arrow; None = fastest ice in each image
ARROW_COLOR = "#c9efff"  # very light blue arrow fill
ARROW_EDGE_COLOR = "black"  # black outline
SHOW_SPEED_COLOUR = False           # True = also draw the pale speed colour layer and its colour bar
VEL_ALPHA = 0.5                     # (only if SHOW_SPEED_COLOUR) transparency of the colour layer
VEL_MAX = None                      # (only if SHOW_SPEED_COLOUR) m/yr at the top of the colour scale
VEL_CMAP = "YlGnBu"                 # (only if SHOW_SPEED_COLOUR)

# ---------- Technical ----------
MAX_PIXELS = 3000
SAVE_PNG = True
SHOW_FIGURES = True
DPI = 200

## 5.2 Load the tools
Nothing to change here. Run the cell; it only defines functions.

In [ ]:
%matplotlib inline
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # avoids HDF errors on network / mounted disks
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import xarray as xr
import geopandas as gpd
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import Affine
from rasterio.warp import calculate_default_transform, reproject
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, ListedColormap
from matplotlib.patches import Rectangle
import matplotlib.patheffects as pe

DST_CRS = "EPSG:3031"          # polar stereographic, same as the velocity data
OUT_ROOT = Path(OUTPUT_DIR).expanduser()

# ------------------------------------------------------------------ files
def _search_folders():
    here = Path.cwd()
    return [Path("~/data").expanduser(), Path("~").expanduser(), here, here / "data", here.parent / "data",
            Path("/gws/ssde/j25b/iecdt/amorgan/data"), Path("/gws/ssde/j25b/iecdt/amorgan")]

def find_file(given, patterns, what):
    if given:
        p = Path(given).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"{what}: {p} does not exist")
        return p
    for folder in _search_folders():
        for pat in patterns:
            hits = sorted(folder.glob(pat)) if folder.exists() else []
            if hits:
                return hits[-1]
    for folder in _search_folders()[:1]:           # last resort: look inside ~/data sub-folders
        for pat in patterns:
            hits = sorted(folder.glob("**/" + pat)) if folder.exists() else []
            if hits:
                return hits[-1]
    raise FileNotFoundError(f"{what} not found. Set it by hand in 5.1.")

def open_netcdf(path):
    """Open a NetCDF file; give a clear message if the file is not readable."""
    path = Path(path)
    size = path.stat().st_size
    with open(path, "rb") as f:
        head = f.read(8)
    if not (head[:4] == b"\x89HDF" or head[:3] == b"CDF"):
        raise ValueError(f"{path.name} is not a NetCDF file ({size / 1e6:.1f} MB, starts with {head!r}). "
                         "It is probably a failed download (e.g. a login page). Download it again.")
    errors = []
    for engine in ("netcdf4", "h5netcdf"):
        try:
            return xr.open_dataset(path, engine=engine)
        except Exception as e:
            errors.append(f"{engine}: {type(e).__name__}: {e}")
    raise OSError(f"Cannot read {path.name} ({size / 1e6:.1f} MB). The file is probably incomplete or damaged: "
                  "compare its size with the one on the NSIDC page and download it again.\n" + "\n".join(errors))

# ------------------------------------------------------------------ images
def load_geo(tif, max_px=MAX_PIXELS):
    """Read a Step 3/4 GeoTIFF (GCPs or a normal georeference) and warp it to EPSG:3031."""
    with rasterio.open(tif) as src:
        h, w, n = src.height, src.width, src.count
        f = max(1, int(np.ceil(max(h, w) / max_px)))
        oh, ow = int(np.ceil(h / f)), int(np.ceil(w / f))
        sx, sy = w / ow, h / oh
        data = src.read(out_shape=(n, oh, ow), resampling=Resampling.average).astype("float32")
        dtype, nodata = src.dtypes[0], src.nodata
        gcps, gcrs = src.gcps
        if gcps:
            gcps = [GroundControlPoint(row=g.row / sy, col=g.col / sx, x=g.x, y=g.y, z=g.z) for g in gcps]
            gcrs = gcrs or "EPSG:4326"
            src_kw = dict(gcps=gcps, src_crs=gcrs)
            tf, W, H = calculate_default_transform(gcrs, DST_CRS, ow, oh, gcps=gcps)
            if len(gcps) >= 10:
                src_kw["tps"] = True          # smoother fit when there are enough GCPs
        elif src.crs is not None:
            tr = src.transform * Affine.scale(sx, sy)
            src_kw = dict(src_transform=tr, src_crs=src.crs)
            tf, W, H = calculate_default_transform(src.crs, DST_CRS, ow, oh, left=tr.c, top=tr.f,
                                                   right=tr.c + tr.a * ow, bottom=tr.f + tr.e * oh)
        else:
            raise ValueError(f"{tif.name} has neither GCPs nor a georeference")
    out = np.full((n, H, W), np.nan, "float32")
    extra = {"src_nodata": nodata} if nodata is not None else {}
    try:
        reproject(data, out, dst_transform=tf, dst_crs=DST_CRS, resampling=Resampling.bilinear,
                  dst_nodata=np.nan, **src_kw, **extra)
    except TypeError:                          # older rasterio without tps
        src_kw.pop("tps", None)
        reproject(data, out, dst_transform=tf, dst_crs=DST_CRS, resampling=Resampling.bilinear,
                  dst_nodata=np.nan, **src_kw, **extra)
    return dict(data=out, tf=tf, H=H, W=W, dtype=dtype, decimation=f,
                left=tf.c, top=tf.f, right=tf.c + tf.a * W, bottom=tf.f + tf.e * H)

def pauli_rgb(geo):
    """RGBA array (H, W, 4) and a validity mask for a Pauli GeoTIFF."""
    d = geo["data"]
    rgb = d[:3].copy()
    valid = np.isfinite(rgb).all(0) & (np.nan_to_num(rgb).sum(0) > 0)
    if d.shape[0] >= 4:
        valid &= np.nan_to_num(d[3]) > 0
    if geo["dtype"] == "uint8":
        rgb /= 255.0
    elif np.nanmax(rgb[:, valid]) > 1.5:
        for i in range(3):
            lo, hi = np.nanpercentile(rgb[i][valid], [2, 98])
            rgb[i] = (rgb[i] - lo) / max(hi - lo, 1e-12)
    rgba = np.dstack([np.clip(np.nan_to_num(rgb.transpose(1, 2, 0)), 0, 1), valid.astype("float32")])
    return rgba, valid

# CP band: (index in the 6-band tif, label, colour map, vmin, vmax)
CP_BANDS = {
    "span":  (0, "span (dB)",    "gray",    None, None),
    "H":     (1, "entropy H",    "viridis", 0, 1),
    "A":     (2, "anisotropy A", "viridis", 0, 1),
    "alpha": (3, "mean alpha (deg)", "plasma", 0, 90),
    "beta":  (4, "mean beta (deg)",  "plasma", 0, 90),
    "beta1": (5, "beta 1 (deg)",     "plasma", 0, 90),
}

def cp_layer(geo, name):
    """One Cloude-Pottier map as a masked array plus its colour settings."""
    idx, label, cmap, vmin, vmax = CP_BANDS[name]
    d = geo["data"]
    band = d[idx].copy()
    if name == "span":
        band = 10 * np.log10(np.where(band > 0, band, np.nan))
    if name == "A":
        band[d[1] < 0.3] = np.nan          # A is not meaningful where H < 0.3 (as in Step 4)
    if name in ("beta", "beta1"):
        band[d[3] < 15] = np.nan           # beta undefined for pure surface scattering
    valid = np.isfinite(d[1]) & (np.nan_to_num(d[1]) > 0) if d.shape[0] > 1 else np.isfinite(band)
    if vmin is None:
        vmin, vmax = np.nanpercentile(band[np.isfinite(band)], [2, 98])
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("0.75")                      # grey = not defined
    shown = np.ma.masked_where(~valid, band)       # outside the scene stays transparent
    shown = np.ma.masked_invalid(shown)
    return shown, valid, label, cm, vmin, vmax

# ------------------------------------------------------------------ velocity + ice shelves
def _crop(da, x0, x1, y0, y1):
    xs = slice(x0, x1) if float(da.x[0]) < float(da.x[-1]) else slice(x1, x0)
    ys = slice(y0, y1) if float(da.y[0]) < float(da.y[-1]) else slice(y1, y0)
    return da.sel(x=xs, y=ys).sortby("x").sortby("y")

class VelocityOnShelves:
    """Annual velocity restricted to the MEaSUREs ice-shelf polygons."""
    def __init__(self, vel_file, shelf_file):
        self.ds = open_netcdf(vel_file)
        pick = lambda names: next((n for n in names if n in self.ds.variables), None)
        vx, vy = pick(["VX", "vx", "Vx"]), pick(["VY", "vy", "Vy"])
        if vx is None or vy is None or "x" not in self.ds.coords or "y" not in self.ds.coords:
            raise KeyError(f"Expected VX, VY on an x/y grid; found {list(self.ds.variables)}")
        self.vx, self.vy = self.ds[vx].squeeze(), self.ds[vy].squeeze()
        shelves = gpd.read_file(shelf_file)
        if shelves.crs is not None and shelves.crs.to_epsg() != 3031:
            shelves = shelves.to_crs(DST_CRS)
        self.shelves = shelves[shelves.geometry.notna()]

    def region(self, geo, valid, margin_m=5000):
        """Speed, vx, vy (NaN off the ice shelves / outside the image) on the velocity grid, in km coordinates."""
        x0, x1 = geo["left"] - margin_m, geo["right"] + margin_m
        y0, y1 = geo["bottom"] - margin_m, geo["top"] + margin_m
        vx, vy = _crop(self.vx, x0, x1, y0, y1), _crop(self.vy, x0, x1, y0, y1)
        if vx.size == 0:
            return None
        near = self.shelves.cx[x0:x1, y0:y1]
        if len(near) == 0:
            return None                                    # no ice shelf in this scene
        dx, dy = abs(float(vx.x[1] - vx.x[0])), abs(float(vx.y[1] - vx.y[0]))
        grid = Affine(dx, 0, float(vx.x[0]) - dx / 2, 0, -dy, float(vx.y[-1]) + dy / 2)   # north-up, y sorted ascending
        keep = rasterize(near.geometry, out_shape=vx.shape, transform=grid, all_touched=True, dtype="uint8")
        keep = np.flipud(keep).astype(bool)
        # keep only cells where the image has data (a velocity cell is ~1 km, so use a clear majority)
        res = abs(geo["tf"].a)
        win = max(1, int(round(1000 / res)))
        frac = uniform_filter(valid.astype("float32"), size=win, mode="constant")
        X, Y = np.meshgrid(vx.x.values, vx.y.values)
        col = np.floor((X - geo["left"]) / geo["tf"].a).astype(int)
        row = np.floor((Y - geo["top"]) / geo["tf"].e).astype(int)
        inside = (col >= 0) & (col < geo["W"]) & (row >= 0) & (row < geo["H"])
        covered = np.zeros_like(keep)
        covered[inside] = frac[row[inside], col[inside]] > 0.75
        keep &= covered
        u, v = vx.values.astype("float32"), vy.values.astype("float32")
        u[~keep], v[~keep] = np.nan, np.nan
        return dict(x=vx.x.values / 1000, y=vx.y.values / 1000, u=u, v=v, speed=np.hypot(u, v))

# ------------------------------------------------------------------ drawing
def light_cmap(name, lo=0.05, hi=0.6):
    return ListedColormap(plt.get_cmap(name)(np.linspace(lo, hi, 256)))
VEL_COLORS = light_cmap(VEL_CMAP)

def nice_speed(v):
    """Largest round number (1, 2, 5 x 10^n) that is not above v."""
    if not np.isfinite(v) or v <= 0:
        return 1.0
    p = 10 ** np.floor(np.log10(v))
    return float(max(m for m in (1, 2, 5, 10) if m * p <= v) * p)

def add_velocity(ax, vel, vmax):
    """Arrows scaled by speed (never overlapping) + a scale key. Optional pale colour layer."""
    if vel is None or not np.isfinite(vel["speed"]).any():
        return False
    x, y = vel["x"], vel["y"]
    dx, dy = abs(x[1] - x[0]), abs(y[1] - y[0])
    if SHOW_SPEED_COLOUR:
        ax.imshow(np.ma.masked_invalid(vel["speed"]), origin="lower", cmap=VEL_COLORS, norm=Normalize(0, vmax),
                  alpha=VEL_ALPHA, interpolation="nearest", zorder=2,
                  extent=[x[0] - dx / 2, x[-1] + dx / 2, y[0] - dy / 2, y[-1] + dy / 2])
    step = max(1, int(round(ARROW_SPACING_KM / min(dx, dy))))
    spacing = step * max(dx, dy)                         # real distance between arrow centres (km)
    X, Y = np.meshgrid(x[::step], y[::step])
    U, V, S = vel["u"][::step, ::step], vel["v"][::step, ::step], vel["speed"][::step, ::step]
    ok = np.isfinite(S) & (S > 0)
    X, Y, U, V, S = X[ok], Y[ok], U[ok], V[ok], S[ok]
    if S.size == 0:
        return False
    smax = ARROW_MAX_SPEED or float(S.max())
    # Percentile-based arrow scaling
    valid_speeds = S[np.isfinite(S) & (S > 0)]
    
    if valid_speeds.size == 0:
        return False
    
    # Inspect velocity distribution
    p50, p75, p90, p95, p99 = np.percentile(
        valid_speeds, [50, 75, 90, 95, 99]
    )
    
    print(f"Velocity distribution (m/yr):")
    print(f"  Median: {p50:.1f}")
    print(f"  P75:    {p75:.1f}")
    print(f"  P90:    {p90:.1f}")
    print(f"  P95:    {p95:.1f}")
    print(f"  P99:    {p99:.1f}")
    print(f"  Max:    {valid_speeds.max():.1f}")
    
    # Use 95th percentile as scaling reference
    speed_cap = p95
    
    # Arrow lengths in map coordinates (km)
    max_arrow_length = 0.90 * spacing
    min_arrow_length = 0.25 * spacing
    
    k = max_arrow_length / speed_cap
    
    length = np.clip(
        S * k,
        min_arrow_length,
        max_arrow_length
    )    # Shaft and head are both visible: minimum shaft length is protected by minshaft.
    # A black stroke surrounds the pale-blue filled arrow, including its tail.
    qkw = dict(angles="xy", scale_units="xy", scale=1, pivot="mid",
               color=ARROW_COLOR, width=0.0048, headwidth=5,
               headlength=4.5, headaxislength=3.8, minshaft=2.0,
               minlength=0, linewidth=0)
    arrows = ax.quiver(X, Y, U / S * length, V / S * length, zorder=3, **qkw)
    arrows.set_path_effects([pe.Stroke(linewidth=0.75, foreground=ARROW_EDGE_COLOR), pe.Normal()])
    # scale key (always shown): a reference arrow, drawn with the same scale as the data arrows
    ref = nice_speed(smax)
    u = ref * k
    (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    span = x1 - x0
    ax.add_patch(Rectangle((0.02, 0.02), min(0.95, u / span + 0.20), 0.075, transform=ax.transAxes,
                           facecolor="white", edgecolor="0.5", alpha=0.9, zorder=4))
    cx, cy = x0 + 0.04 * span + u / 2, y0 + 0.0575 * (y1 - y0)
    key_arrow = ax.quiver([cx], [cy], [u], [0], zorder=5, **qkw)
    key_arrow.set_path_effects([pe.Stroke(linewidth=0.65, foreground=ARROW_EDGE_COLOR), pe.Normal()])
    ax.text(cx + u / 2 + 0.012 * span, cy, f"{ref:g} m/yr", va="center", fontsize=8, zorder=5)
    return True

def finish_axes(ax, geo, valid):
    rows, cols = np.where(valid)
    tf = geo["tf"]
    xs = [tf.c + tf.a * cols.min(), tf.c + tf.a * (cols.max() + 1)]
    ys = [tf.f + tf.e * (rows.max() + 1), tf.f + tf.e * rows.min()]
    padx, pady = 0.02 * (xs[1] - xs[0]), 0.02 * (ys[1] - ys[0])
    ax.set_xlim((xs[0] - padx) / 1000, (xs[1] + padx) / 1000)
    ax.set_ylim((ys[0] - pady) / 1000, (ys[1] + pady) / 1000)
    ax.set_aspect("equal")
    ax.set_facecolor("0.93")
    ax.set_xlabel("Polar stereographic x (km)")
    ax.set_ylabel("Polar stereographic y (km)")

def extent_km(geo):
    return [geo["left"] / 1000, geo["right"] / 1000, geo["bottom"] / 1000, geo["top"] / 1000]

def make_figure(tif, kind, vmodel, out_png):
    product = tif.stem.rsplit("_", 2)[0] if kind == "h_alpha" else tif.stem.rsplit("_", 1)[0]
    geo = load_geo(tif)
    if kind == "pauli":
        rgba, valid = pauli_rgb(geo)
        layers = [("pauli", rgba)]
        n = 1
    else:
        layers = []
        for name in CP_PANELS:
            shown, valid, label, cm, vmin, vmax_b = cp_layer(geo, name)
            layers.append((name, (shown, label, cm, vmin, vmax_b)))
        n = len(layers)
    if not valid.any():
        print("   ! no valid image data, skipped")
        return
    vel = vmodel.region(geo, valid)
    vmax = VEL_MAX or (np.nanpercentile(vel["speed"], 98) if vel is not None and np.isfinite(vel["speed"]).any() else 1)
    fig, axes = plt.subplots(1, n, figsize=(8.5 * n if n > 1 else 9, 8.5), layout="constrained", squeeze=False)
    axes = axes[0]
    drew = False
    for ax, (name, payload) in zip(axes, layers):
        if kind == "pauli":
            ax.imshow(payload, extent=extent_km(geo), origin="upper", zorder=1)
            ax.set_title("Pauli RGB  (R |HH-VV|, G |HV+VH|, B |HH+VV|)", fontsize=10)
        else:
            shown, label, cm, vmin, vmax_b = payload
            im = ax.imshow(shown, extent=extent_km(geo), origin="upper", cmap=cm, vmin=vmin, vmax=vmax_b, zorder=1)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=label)
            ax.set_title(label, fontsize=10)
        finish_axes(ax, geo, valid)
        drew = add_velocity(ax, vel, vmax) or drew
    if drew and SHOW_SPEED_COLOUR:
        fig.colorbar(ScalarMappable(Normalize(0, vmax), VEL_COLORS), ax=list(axes), location="bottom",
                                          shrink=0.45, aspect=40, pad=0.02, label="Ice speed on ice shelves (m/yr)")
    note = "" if drew else "   (no ice shelf with velocity data in this scene)"
    fig.suptitle(f"{product}{note}", fontsize=10)
    if SAVE_PNG:
        out_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)

# ------------------------------------------------------------------ find the Step 3/4 results
def find_results(kind):
    """Newest GeoTIFF of every scene: {path relative to its run folder: (tif, run folder name)}."""
    found = {}
    for tif in sorted(OUT_ROOT.glob(f"*/**/*_{kind}.tif")):
        rel = tif.relative_to(OUT_ROOT)
        run, inner = rel.parts[0], Path(*rel.parts[1:])
        if run.endswith("velocity_overlay"):
            continue
        if RUN_FOLDERS and not any(r in run for r in RUN_FOLDERS):
            continue
        if STACKS and not any(s in "/".join(inner.parts) for s in STACKS):
            continue
        found[inner] = (tif, run)       # sorted by run name (= time stamp), so the newest wins
    return found

def run_overlay(kind, label):
    results = find_results(kind)
    print(f"{label}: {len(results)} scene(s) found")
    for i, (inner, (tif, run)) in enumerate(sorted(results.items()), 1):
        print(f"[{i}/{len(results)}] {inner.parent}  (from run {run})")
        try:
            make_figure(tif, kind, vmodel, RUN_DIR / inner.parent / f"{tif.stem}_velocity.png")
        except Exception as e:
            print(f"   ! failed: {type(e).__name__}: {e}")

## 5.3 Check the inputs
Shows which files and results were found. If a stack is listed under "no result found", run Step 3 / Step 4 for it first.

In [ ]:
vel_path = find_file(VELOCITY_FILE, ["Antarctica_ice_velocity_2024_2025_1km_v01.2.nc", "Antarctica_ice_velocity*2024*2025*.nc"], "Velocity file")
shelf_path = find_file(ICE_SHELF_FILE, ["IceShelf_Antarctica_v02.shp", "IceShelf_Antarctica*.shp"], "Ice-shelf shapefile (MEaSUREs)")
print("velocity    :", vel_path)
print("ice shelves :", shelf_path)

vmodel = VelocityOnShelves(vel_path, shelf_path)
print(f"velocity grid: {vmodel.vx.sizes['x']} x {vmodel.vx.sizes['y']} cells, units m/yr, CRS {DST_CRS}\n")

for kind, label in [("pauli", "Pauli"), ("h_alpha", "Cloude-Pottier")]:
    res = find_results(kind)
    print(f"{label}: {len(res)} result(s)")
    for s in (STACKS or []):
        n = sum(s in "/".join(k.parts) for k in res)
        print(f"   {'ok ' if n else 'no result found'}  {n:2d}  {s}")
    print()

## 5.4 Pauli + velocity
One figure per scene. Pale colours = ice speed on ice shelves, arrows = flow direction.

In [ ]:
RUN_DIR = OUT_ROOT / f"{datetime.now():%Y%m%d_%H%M%S}_velocity_overlay"
if DO_PAULI:
    run_overlay("pauli", "Pauli")

## 5.5 Cloude–Pottier + velocity
Same overlay on the maps chosen in `CP_PANELS` (default: entropy H and mean α). Grey inside a scene = not defined (A where H < 0.3, β where α < 15°).

In [ ]:
if DO_CLOUDE_POTTIER:
    run_overlay("h_alpha", "Cloude-Pottier")

RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "run_info.json").write_text(json.dumps(dict(
    velocity_file=str(vel_path), ice_shelf_file=str(shelf_path), stacks=STACKS, run_folders=RUN_FOLDERS,
    cp_panels=CP_PANELS, arrow_spacing_km=ARROW_SPACING_KM, arrow_max_speed=ARROW_MAX_SPEED,
    show_speed_colour=SHOW_SPEED_COLOUR, max_pixels=MAX_PIXELS), indent=2))
print("Results in:", RUN_DIR)

## Troubleshooting
- **"Ice-shelf shapefile not found"**: download *MEaSUREs Antarctic Boundaries for IPY 2007-2009, v2* (NSIDC-0709) and point `ICE_SHELF_FILE` to `IceShelf_Antarctica_v02.shp` (keep the other files with the same name next to it).
- **"no result found"** for a stack: run Step 3 / Step 4 with `SAVE_GEOTIFF = True` and the same `OUTPUT_DIR`.
- **Overlay is shifted against the image**: check the first lines of 5.3; the images are warped from their GCPs, so a shift usually means the GCPs in the `.tif` are wrong, not the velocity.
- **No colours / arrows in a scene**: the scene has no ice shelf with velocity data (e.g. only grounded ice or open ocean).
- **Kernel runs out of memory**: lower `MAX_PIXELS` (e.g. 2000).